# Streaming Productivity Assistant — Practical Walkthrough

This notebook supports the full-stack practical by isolating the backend ideas before the React dashboard is opened.

You will see how a mock productivity event becomes:

1. a validated event payload,
2. an assistant prompt,
3. a stream of response chunks,
4. a persisted event/run/chunk history in SQLite.

The dashboard version uses the same backend files, then adds WebSocket delivery to the browser.

## Setup

Run this notebook from the project root so Python can import the `backend` package.

In [ ]:
from pathlib import Path
import asyncio
import json
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

## Load a sample productivity event

The practical uses mock events instead of real Gmail/Calendar credentials. This keeps the class focused on event-driven agent design rather than OAuth setup.

In [ ]:
from backend.mock_events import SAMPLE_EVENTS

sample_event = SAMPLE_EVENTS[0]
sample_event.model_dump()

## Convert the event into an assistant prompt

The backend does not send raw frontend state directly to the model. It builds a controlled prompt from a validated event.

In [ ]:
from backend.agent import build_event_prompt

prompt = build_event_prompt(sample_event.model_dump())
print(prompt)

## Stream chunks locally

The classroom default is mock mode. This gives a predictable token-by-token experience without spending API credits.

In [ ]:
from backend.agent import stream_mock_response

async for chunk in stream_mock_response(sample_event.model_dump()):
    print(chunk, end="")

## Persist the event and run in SQLite

A real-time UI still needs durable state. If the page refreshes, persisted chunks can be replayed.

In [ ]:
from backend.database import (
    append_chunk,
    create_event,
    create_run,
    get_run,
    init_db,
    list_chunks,
    reset_demo_data,
    update_run_status,
)

init_db()
reset_demo_data()

stored_event = create_event(sample_event)
run = create_run(stored_event["id"])

stored_event, run

## Simulate an end-to-end backend run

This mirrors what the FastAPI background task does: stream chunks, save each chunk, and save the final output.

In [ ]:
from backend.agent import stream_productivity_response

full_output = ""
sequence = 0

update_run_status(run["id"], "running")

async for chunk in stream_productivity_response(stored_event):
    sequence += 1
    append_chunk(run["id"], sequence, chunk)
    full_output += chunk

update_run_status(run["id"], "completed", final_output=full_output)

print(full_output)

## Inspect persisted chunks

The WebSocket endpoint replays these rows to a browser that connects late or refreshes mid-run.

In [ ]:
chunks = list_chunks(run["id"])
print(f"Persisted chunks: {len(chunks)}")
chunks[:5]

## Run the full app after the notebook

Backend:

```bash
uvicorn backend.main:app --reload --host 127.0.0.1 --port 8000
```

Frontend:

```bash
cd frontend
npm install
npm run dev
```

Then open the Vite URL and trigger a mock event.

## Challenge exercises

1. Add a new event type called `slack`.
   - Hint: update the event schema type, mock event list, and mock response logic.

2. Add a retry button for failed runs.
   - Hint: create a new run for the same event rather than overwriting the old run.

3. Batch chunks before writing to SQLite.
   - Hint: collect every 5 chunks and write a larger segment.

## Closing summary

You now have the core pattern for real-time agent UX:

`event trigger → persistent run → assistant stream → WebSocket fan-out → dashboard replay`

This pattern is reusable for support copilots, calendar assistants, monitoring agents, and operations dashboards.